In [1]:
pip install beautifulsoup4 youtube-transcript-api requests typing

In [2]:
import re
import logging
from typing import List, Optional
import requests
from bs4 import BeautifulSoup
from youtube_transcript_api import YouTubeTranscriptApi

In [ ]:


class ContentError(Exception):
    """Custom exception for content fetching errors."""
    pass

def fetch_wikipedia_content(wiki_url: str) -> str:

    try:
        # Validate Wikipedia URL
        if not re.match(r'https?://[a-z]+\.wikipedia\.org/wiki/', wiki_url):
            raise ContentError("Invalid Wikipedia URL format")

        # Fetch the page
        response = requests.get(wiki_url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove unwanted sections
        for unwanted in soup.find_all(['table', 'script', 'style', 'sup', 'span.mw-editsection']):
            unwanted.decompose()

        # Get the main content
        content_div = soup.find(id='mw-content-text')
        if not content_div:
            raise ContentError("Could not find main content")

        # Extract paragraphs
        paragraphs = content_div.find_all('p')

        # Clean and join the text
        content = ' '.join(
            p.get_text().strip()
            for p in paragraphs
            if p.get_text().strip()  # Skip empty paragraphs
        )

        # Clean up special characters and extra whitespace
        content = re.sub(r'\[\d+\]', '', content)  # Remove reference numbers
        content = re.sub(r'\s+', ' ', content)  # Normalize whitespace

        if not content:
            raise ContentError("No content found in the article")

        return content

    except requests.exceptions.RequestException as e:
        raise ContentError(f"Failed to fetch Wikipedia page: {str(e)}")
    except Exception as e:
        raise ContentError(f"Error processing Wikipedia content: {str(e)}")

def fetch_youtube_transcript(video_url: str) -> str:
    """Existing YouTube transcript fetching function"""
    # Your existing fetch_transcript function code here
    try:
        video_id_match = re.search(
            r"(?:v=|\/|youtu\.be\/|embed\/)([a-zA-Z0-9_-]{11})",
            video_url
        )
        if not video_id_match:
            raise ContentError("Invalid YouTube URL format")

        video_id = video_id_match.group(1)

        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            try:
                transcript = transcript_list.find_transcript(['en'])
            except:
                transcript = transcript_list.find_generated_transcript(['en'])

            transcript_data = transcript.fetch()
            return " ".join(item['text'] for item in transcript_data)

        except Exception as e:
            available_transcripts = []
            try:
                available_transcripts = [
                    t.language_code
                    for t in transcript_list.manual_transcripts
                ]
            except:
                pass

            raise ContentError(
                f"Failed to fetch transcript. Available languages: {available_transcripts}. Error: {str(e)}"
            )

    except Exception as e:
        raise ContentError(f"Error processing video: {str(e)}")

def preprocess_content(content: str, chunk_size: int = 200) -> List[str]:
    """
    Preprocess content into chunks with improved text splitting.
    Works for both Wikipedia and YouTube content.

    Args:
        content: Input text content
        chunk_size: Maximum size of each chunk

    Returns:
        List of text chunks
    """
    # Split on sentence boundaries
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', content) if s.strip()]
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)
        if current_length + sentence_length <= chunk_size:
            current_chunk.append(sentence)
            current_length += sentence_length
        else:
            if current_chunk:
                chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
            current_length = sentence_length

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

def query_llama_model(api_key: str, prompt: str) -> Optional[str]:
    """Existing Groq API query function"""
    if not api_key:
        raise ValueError("API key is required")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful assistant that answers questions about content based on provided context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.7
    }

    try:
        response = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:
            logging.error(f"API Error: {response.status_code} - {response.text}")
            return None

        result = response.json()
        if not result.get("choices"):
            raise ValueError("No choices in response")

        return result["choices"][0]["message"]["content"].strip()

    except requests.exceptions.RequestException as e:
        logging.error(f"API request failed: {str(e)}")
        return None
    except (KeyError, ValueError) as e:
        logging.error(f"Error parsing API response: {str(e)}")
        return None

def content_qa_system(url: str, question: str, api_key: str) -> str:
    """
    Main QA system that handles both Wikipedia and YouTube URLs.

    Args:
        url: Wikipedia or YouTube URL
        question: User question
        api_key: Groq API key

    Returns:
        Answer string or error message
    """
    try:
        # Input validation
        if not url or not question or not api_key:
            raise ValueError("Missing required parameters")

        # Determine content type and fetch accordingly
        if 'wikipedia.org' in url:
            content = fetch_wikipedia_content(url)
            content_type = "Wikipedia article"
        elif 'youtu' in url:  # Handles youtube.com and youtu.be
            content = fetch_youtube_transcript(url)
            content_type = "video transcript"
        else:
            raise ValueError("Unsupported URL type. Please provide a Wikipedia or YouTube URL.")

        if not content:
            return f"Could not extract content from {content_type}"

        # Process content
        chunks = preprocess_content(content)
        if not chunks:
            return "Failed to process content"

        # Prepare prompt
        context = " ".join(chunks)
        prompt = (
            f"Based on the following {content_type}, please answer the question.\n\n"
            f"Content: {context}\n\n"
            f"Question: {question}\n\n"
            f"Answer:"
        )

        # Get model response
        answer = query_llama_model(api_key, prompt)
        if not answer:
            return "Failed to generate answer"

        return answer

    except Exception as e:
        logging.error(f"Error in QA system: {str(e)}")
        return f"An error occurred: {str(e)}"

# Example usage
if __name__ == "__main__":
    print("select functionality")
    print("1. Wikipedia page")
    print("2. Youtube Video ")
    choice = input("Enter 1 or 2: ")
    if choice == "1":
        link = input("Paste Wikipedia page link: ")
        question = input("Enter your question: ")
        #process_wikipedia(link, question)
    elif choice == "2":
        link = input("Paste YouTube video link: ")
        question = input("Enter your question: ")
        #process_youtube(link, question)
    else:
        print("Invalid choice. Please restart and enter 1 or 2.")

    #url = "https://youtu.be/eBSeCp__xhI?feature=shared"
    #question ="Give me a list of points that author told and that I should apply in my life."
    groq_api_key = ""  # Replace with actual API key

    logging.basicConfig(level=logging.INFO)
    answer = content_qa_system(url, question, groq_api_key)
    print("Answer:", answer)

select functionality
1. Wikipedia page
2. Youtube Video 
Enter 1 or 2: 2
Paste YouTube video link: https://youtu.be/eBSeCp__xhI?feature=shared
Enter your question: What is this video about? What are the things from the video that I should apply in my life?
Answer: Based on the provided video transcript, this video is about the importance of perseverance, determination, and a relentless attitude in achieving success and reaching one's destiny. The speaker emphasizes the need to overcome obstacles, setbacks, and discouragement to pursue one's goals and dreams.

There are several key takeaways from the video that you can apply in your life:

1. **Made-up mind**: Develop a mindset that is unwavering and unshakeable. Believe in yourself and your goals, even when faced with challenges and setbacks.
2. **Relentless pursuit**: Be passionate and dedicated to your goals. Don't give up easily, even when faced with obstacles or people who tell you no.
3. **Focus on the mental preparation**: Unders